# Wind-up example using the SMARTEOLE dataset

Dataset citation:
Thomas Duc, & Eric Simley. (2022): [SMARTEOLE Wind Farm Control open dataset (1.0)](https://doi.org/10.5281/zenodo.7342466)

SMARTEOLE is a wake steering toggle test: SMV6 steers its wake away from SMV5 when the wind blows
from 207 to 236 degrees, and the controller switches steering on and off in turn.

This notebook uses the wind-up v1 interface, which describes an analysis in three layers:

1. **Facts** (`WindFarmFacts`): what is true of the wind farm and what was done to it: the SCADA
   records, which column means what, the turbines, the upgrade and its toggle signal, the northing
   table, the analyst's labels.
2. **Analysis** (`AnalysisSpec`): the question asked of the facts: which upgrade, which turbines,
   against which references, over which records.
3. **Method**: answers analyses from the facts, `method.estimate(facts, analyses)`, with options
   that are its own.

The same facts and analysis are answered by two methods: `toggle_energy_ratio` (new in v1) and the
v0 binned power curve method of wind-up 0.7.

In [1]:
import logging
from pathlib import Path

import pandas as pd
from helpers import download_zenodo_data, setup_logger

import wind_up

setup_logger()
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
logger.info("wind-up version: %s", wind_up.__version__)

DATA_DIR = Path.cwd().parent / "tests" / "test_data" / "smarteole"
OUT_DIR = Path.home() / "wind-up-output" / "smarteole_example"
pd.set_option("display.width", 200)

wind-up version: 0.7.2


## 1. Download the open dataset

In [2]:
FILENAME = "SMARTEOLE-WFC-open-dataset.zip"
download_zenodo_data(
    record_id="7342466",
    output_dir=DATA_DIR,
    filenames=[FILENAME],
    cache_overwrite=False,  # change to True to force the download
)

All filenames are locally cached, no download necessary.


## 2. Extract and format the data

The 1-minute data is averaged to 10 minutes, the analysis timebase.

In [3]:
from smarteole_utils import SmartEoleExtractor

TIMEBASE = pd.Timedelta(minutes=10)

extractor = SmartEoleExtractor.from_zip_file(
    zipped_fpath=DATA_DIR / FILENAME,
    analysis_timebase_s=int(TIMEBASE.total_seconds()),
)

#### 2.1 SCADA records

One row per turbine and timestamp, timestamps in UTC at the start of each period. The columns keep
the dataset's own names; section 3.2 tells wind-up what each one means.

SMARTEOLE has no availability counter, but it flags each minute of suspected derating per turbine.
The helper turns the flag into one: `available_s` is the record's length less 60 s for each
derated minute.

In [4]:
scada = extractor.unpack_smarteole_scada()
scada.head(3)

,turbine,active_power_avg,active_power_std,wind_speed_avg,wind_speed_std,nacelle_position_avg,nacelle_position_min,nacelle_position_max,blade_1_pitch_angle_avg,generator_speed_avg,temperature_avg,available_s
timestamp,,,,,,,,,,,,
2020-02-17 16:30:00+00:00,SMV1,2017.8566,57.0837,13.6164,1.2411,247.884662,239.930,259.686,6.0648,1800.5394,11.6191,600.0
2020-02-17 16:40:00+00:00,SMV1,1946.2472,91.7614,13.0139,1.0178,259.686000,259.686,259.686,3.5153,1798.7652,11.5493,600.0
2020-02-17 16:50:00+00:00,SMV1,1946.5069,98.3934,12.5818,0.9482,259.686000,259.686,259.686,2.1367,1799.5171,11.4929,600.0


#### 2.2 Turbine positions

In [5]:
metadata = extractor.unpack_smarteole_metadata()
metadata.head(3)

,Turbine,Latitude,Longitude
0,SMV1,49.84975,2.80194
1,SMV2,49.84661,2.80150
2,SMV3,49.84347,2.80162


#### 2.3 Toggle signal

The controller's log, one row per timestamp: `toggle_on` while steering is active, `toggle_off`
while it is not. A record in neither state (a switch part way through the period) is in neither.

In [6]:
toggle = extractor.unpack_smarteole_toggle_data()
toggle[toggle["yaw_offset_command"] > 10].head(3)  # noqa: PLR2004

,toggle_on,toggle_off,yaw_offset_command
timestamp,,,
2020-02-18 07:00:00+00:00,True,False,10.1839
2020-02-18 07:30:00+00:00,True,False,10.4246
2020-02-18 07:40:00+00:00,True,False,10.9057


## 3. The facts

#### 3.1 The turbines

Position, rated power and rotor diameter of each turbine.

In [7]:
turbines = pd.DataFrame(
    {
        "name": metadata["Turbine"],
        "latitude": metadata["Latitude"],
        "longitude": metadata["Longitude"],
        "rated_power_kw": 2050.0,
        "rotor_diameter_m": 82.0,
    }
)
turbines

,name,latitude,longitude,rated_power_kw,rotor_diameter_m
0,SMV1,49.84975,2.80194,2050.0,82.0
1,SMV2,49.84661,2.80150,2050.0,82.0
2,SMV3,49.84347,2.80162,2050.0,82.0
3,SMV4,49.84033,2.80122,2050.0,82.0
4,SMV5,49.83719,2.80080,2050.0,82.0
5,SMV6,49.83477,2.79886,2050.0,82.0
6,SMV7,49.83196,2.79934,2050.0,82.0


#### 3.2 Which column means what

wind-up assumes no column names: the schema names the column that plays each role. Only the
turbine and active power columns are required; a method checks the roles it reads.

In [8]:
from wind_up.schema import ColumnSchema

columns = ColumnSchema(
    turbine="turbine",
    active_power="active_power_avg",
    wind_speed="wind_speed_avg",
    wind_speed_sd="wind_speed_std",
    gen_rpm="generator_speed_avg",
    pitch="blade_1_pitch_angle_avg",
    nacelle_position="nacelle_position_avg",
    ambient_temp="temperature_avg",
    availability="available_s",
)

#### 3.3 Downtime

Each record gets an operating state from generic rules: missing, full or partial downtime, normal
operation. A record with less than its full length available is partial downtime, and one with
none available is full downtime. A parked rotor, recognised by its pitch, is also partial downtime:
here, above 35 degrees, the top of the turbine's normal pitch range. It catches stops the derate
flag does not.

In [9]:
from wind_up.operating_state import OperatingStateConfig

operating_states = OperatingStateConfig(parked_pitch_above_deg=35.0)

#### 3.4 The upgrade

Wake steering changes both turbines of the steer: SMV6 yaws away and SMV5 sits in its deflected
wake. The toggle frame is the timing. Records starting within 2 minutes of a switch between on and
off are in neither state, while the turbine settles.

In [10]:
from wind_up import Upgrade

wake_steering = Upgrade(
    name="wake steering",
    turbines=["SMV6", "SMV5"],
    timing=toggle,
    settling_time=pd.Timedelta(minutes=2),
)
f"toggle cycle: {wake_steering.cycle}"

'toggle cycle: 0 days 02:00:00'

#### 3.5 The northing table

`(turbine, from, offset in degrees)`: the offset added to each turbine's nacelle position from
that time on, to read as a true direction.

In [11]:
FIRST = pd.Timestamp("2020-02-17 16:30:00+0000")
north_offsets = [
    ("SMV1", FIRST, 5.750994540354649),
    ("SMV2", FIRST, 5.690999999999994),
    ("SMV3", FIRST, 5.558000000000042),
    ("SMV4", FIRST, 5.936999999999996),
    ("SMV5", FIRST, 6.797253350869262),
    ("SMV6", FIRST, 5.030130916842758),
    ("SMV7", FIRST, 4.605999999999972),
]

#### 3.6 A label: the steering sector

The question is about the records in which the controller steers. That is an analyst's label, a
boolean column of the records. The direction it is taken on must not come from a subject turbine,
since steering changes the subjects' nacelle positions; here it is the reference SMV7's northed
nacelle position. The sector is the controller's, 207 to 236 degrees, widened by 10 degrees on each
side as in the wind-up 0.7 version of this example.

In [12]:
from wind_up import Label
from wind_up.labels import direction_sector

SMV7_OFFSET = {t: offset for t, _, offset in north_offsets}["SMV7"]
smv7_direction = (scada.loc[scada["turbine"] == "SMV7", "nacelle_position_avg"] + SMV7_OFFSET) % 360
in_sector = direction_sector(smv7_direction, sector=(207 - 10, 236 + 10))

# a farm-wide label repeats its value on every turbine's record of that timestamp
scada["in steering sector"] = in_sector.reindex(scada.index).fillna(value=False).to_numpy(dtype=bool)
steering_sector = Label(name="in steering sector", column="in steering sector")
f"{in_sector.mean():.0%} of timestamps are in the steering sector"

'19% of timestamps are in the steering sector'

#### 3.7 Reanalysis data

ERA5 near the site, previously downloaded. The binned power curve method reads it; the toggle
energy ratio method does not.

In [13]:
from wind_up import Reanalysis

reanalysis = Reanalysis(
    name="ERA5T_50.00N_2.75E_100m_1hr",
    latitude=50.0,
    longitude=2.75,
    records=pd.read_parquet(DATA_DIR / "ERA5T_50.00N_2.75E_100m_1hr_20200201_20200531.parquet"),
)

#### 3.8 All the facts together

In [14]:
from wind_up import WindFarmFacts

facts = WindFarmFacts(
    records=scada,
    columns=columns,
    timebase=TIMEBASE,
    turbines=turbines,
    upgrades=[wake_steering],
    operating_states=operating_states,
    north_offsets=north_offsets,
    reanalysis=[reanalysis],
    labels=[steering_sector],
)

Every method sees the same labelled records: each record's operating state, whether it is valid
for uplift, its state under each upgrade and the analyst's labels.

In [15]:
from wind_up.labels import upgrade_state_column

record_labels = facts.record_labels()
hours = record_labels.pivot_table(index="turbine", columns=upgrade_state_column("wake steering"), aggfunc="size") / 6
hours.round(1)

upgrade_state: wake steering,neither,off,on
turbine,,,
SMV1,NaN,2335.5,NaN
SMV2,NaN,2335.5,NaN
SMV3,NaN,2335.5,NaN
SMV4,NaN,2335.5,NaN
SMV5,505.2,1174.0,656.3
SMV6,505.2,1174.0,656.3
SMV7,NaN,2335.5,NaN


## 4. The analysis

The net uplift of the two turbines together, against SMV7, over the records in the steering
sector. A result over a direction sector cannot be extrapolated to a year, so only the campaign
uplift is asked for, not the AEP uplift. One analysis per turbine is added beside the pair, so the
pair's uncertainty can be combined from the turbines'.

In [16]:
from wind_up import AnalysisSpec

steer = AnalysisSpec(
    name="SMV6 steering away from SMV5",
    upgrade="wake steering",
    subjects=["SMV6", "SMV5"],
    references=["SMV7"],
    include_labels=["in steering sector"],
    report="campaign",
)
per_turbine = [
    AnalysisSpec(
        name=turbine,
        upgrade="wake steering",
        subjects=[turbine],
        references=["SMV7"],
        include_labels=["in steering sector"],
        report="campaign",
    )
    for turbine in ("SMV6", "SMV5")
]

## 5. Toggle energy ratio

Compares each subject's energy with the steering on and off against the reference's, with a
block bootstrap for the uncertainty. A record of one state is kept only when the other state has a
record within an hour.

In [17]:
%%time
from wind_up.methods import ToggleEnergyRatioMethod

toggle_method = ToggleEnergyRatioMethod(
    out_dir=OUT_DIR / "toggle_energy_ratio",
    save_plots=True,
    pairing_max_gap=pd.Timedelta(hours=1),
)
steer_output, *turbine_outputs = toggle_method.estimate(facts, [steer, *per_turbine])

CPU times: user 29.8 s, sys: 483 ms, total: 30.3 s
Wall time: 31 s


In [18]:
steer_output.per_turbine

,turbine,aep_uplift,aep_sigma,campaign_uplift,campaign_sigma,actual_energy,hours_on,hours_off
0,SMV6,NaN,NaN,-0.008184,0.005243,1.010469e+06,148.833333,154.666667
1,SMV5,NaN,NaN,0.035155,0.018122,8.259140e+05,139.500000,146.666667


In [19]:
for warning in steer_output.warnings:
    logger.info("%s: %s", warning.severity, warning.message)

low: toggle_energy_ratio gives no sigma for a group of subjects; each subject's sigma is in per_turbine


#### Net uplift of the pair

The steering turbine nearly always produces more than the turbine it helps, since it stands in
freer wind, so the net uplift is weighted by energy, not an average of the two percentages.
`wind_up.combine` gives the energy-weighted uplift of several outputs, with the uncertainty as a
range: the turbines' errors taken as independent at one end and as fully correlated at the other
(they share a reference).

In [20]:
net = wind_up.combine(turbine_outputs)
logger.info(
    "net uplift %.1f%% (the pair's own output: %.1f%%)", 100 * net.campaign_uplift, 100 * steer_output.campaign_uplift
)
logger.info(
    "1-sigma %.1f%% if independent, %.1f%% if correlated",
    100 * net.campaign_sigma_independent,
    100 * net.campaign_sigma_correlated,
)

net uplift 1.1% (the pair's own output: 1.1%)


1-sigma 0.8% if independent, 1.1% if correlated


## 6. The v0 binned power curve method

The wind-up 0.7 method, answering the same facts. It reads the availability counter as v0's
shutdown duration, so the derated records leave it too. It keeps its own filters and options: the
direction filter on each reference's own northed direction (the steering sector above), the
detrending and the bin width. It is not yet in the installed package: it moves to
`wind_up.methods` as `binned_power_curve` before wind-up 1.0, and is imported from the repository
until then.

In [21]:
%%time
from benchmarking.baselines.v0_binned import V0BinnedMethod

END = pd.Timestamp("2020-05-25 00:00:00+0000")
LAST = END - TIMEBASE
binned_method = V0BinnedMethod(
    options={
        "require_ref_wake_free": True,
        "detrend_min_hours": 12,  # shorter than the default of 24 because this dataset is small
        "ref_wd_filter": [207 - 10, 236 + 10],
        "filter_all_test_wtgs_together": True,
        "ref_super_wtgs": [],
        "analysis_first_dt_utc_start": FIRST,
        "lt_first_dt_utc_start": FIRST,
        "lt_last_dt_utc_start": LAST,
        "detrend_first_dt_utc_start": FIRST,
        "detrend_last_dt_utc_start": LAST,
        "years_for_lt_distribution": 0,
        "years_for_detrend": 0,
        "ws_bin_width": 1.0,
        "toggle": {
            "name": "wake steering",
            "detrend_data_selection": "use_toggle_off_data",
            "pairing_filter_method": "any_within_timedelta",
            "pairing_filter_timedelta_seconds": 3600,
        },
    },
    turbine_type={
        "turbine_type": "Senvion-MM82-2050",
        "cutout_ws_mps": 25,
        "normal_operation_pitch_range": (-10.0, 35.0),
        "normal_operation_genrpm_range": (250.0, 2000.0),
        "rpm_v_pw_margin_factor": 0.05,
        "pitch_to_stall": False,
    },
    site_name="Sole du Moulin Vieux",
    auto_choose_refs=False,
    out_dir=OUT_DIR / "binned_power_curve",
)
v0_steer = AnalysisSpec(
    name="SMV6 steering away from SMV5, v0",
    upgrade="wake steering",
    subjects=["SMV6", "SMV5"],
    references=["SMV7"],
    period=(FIRST, END),
    report="campaign",
)
(binned_output,) = binned_method.estimate(facts, [v0_steer])

loaded WindUpConfig assessment_name: SMV6 steering away from SMV5, v0


toggle analysis period (UTC): 2020-02-17 16:30 to 2020-05-25 00:00


long term period (UTC): 2020-02-17 16:30 to 2020-05-25 00:00


detrend period (UTC): 2020-02-17 16:30 to 2020-05-25 00:00


running wind_up analysis for SMV6 steering away from SMV5, v0


running load_smart_scada_and_md_from_file for 2020-02-17 16:30:00+00:00 to 2020-05-24 23:50:00+00:00


... (135 outputs left out)

Saved WindUpConfig to ~/wind-up-output/smarteole_example/binned_power_curve/SMV6 steering away from SMV5, v0/config_SMV6 steering away from SMV5, v0.json


##############################################################################
# combine results per test turbine
##############################################################################


ref_list = ['SMV7']


CPU times: user 27.8 s, sys: 150 ms, total: 28 s
Wall time: 28 s


Its per test-reference results (wind-up 0.7, which kept the derated records, gave -1.0 % on SMV6 and +3.1 % on SMV5):

In [22]:
from helpers import format_and_print_results_table

_ = format_and_print_results_table(binned_output.per_reference)

+-----------+-------------+-----------------+----------------------+--------------+-------------+--------------------------+-------------------------+------------------------+
|  turbine  |  reference  |  energy uplift  |  uplift uncertainty  |  uplift P95  |  uplift P5  |  valid hours toggle off  |  valid hours toggle on  |  mean power toggle on  |
+===========+=============+=================+======================+==============+=============+==========================+=========================+========================+
|   SMV6    |    SMV7     |      -1.0%      |         0.6%         |    -1.9%     |    -0.0%    |          132.3           |          135.5          |          1148          |
|   SMV5    |    SMV7     |      3.2%       |         1.3%         |     1.0%     |    5.4%     |          132.8           |          136.7          |          993           |
+-----------+-------------+-----------------+----------------------+--------------+-------------+-----------------------

## 7. The two methods side by side

Both find a small loss on the steering turbine SMV6 and put the gain on SMV5, the turbine steered
around, and agree within their sigmas. The wind-up 0.7 version of this example, which kept the
derated records, gave a net uplift of +0.8 % for the pair from the binned method's results.

In [23]:
side_by_side = pd.concat(
    {
        "toggle_energy_ratio": steer_output.per_turbine.set_index("turbine"),
        "binned power curve (v0)": binned_output.per_turbine.set_index("turbine"),
    },
    axis=1,
).loc[:, (slice(None), ["campaign_uplift", "campaign_sigma", "hours_on", "hours_off"])]
side_by_side.style.format("{:.1%}", subset=(slice(None), (slice(None), ["campaign_uplift", "campaign_sigma"]))).format(
    "{:.0f}", subset=(slice(None), (slice(None), ["hours_on", "hours_off"]))
)